### 1. Calculo de metricas
Para este notebook se requiere utilizar hasta Python 3.10, para que la libreria AligScore se deben contar con versiones especificas que pueden hacer funcionar mal los procesos de fine tuning por lo que se dejan por separado, la idea es tomar todos los archivos csv que cuentan con el texto cientifico, texto resumen original y el resumen generado por medio del LLM en este notebook y realizar el calculo de las metricas: Legibilidad, Relevancia y Factualidad.

Instalacion libreria AlignScore:
https://github.com/yuh-zha/AlignScore

In [1]:
#!pip install --quiet  -r req-fine-models-metrics.txt
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [2]:
import pandas as pd
import numpy as np
import textstat
from typing import List, Dict, Any, Optional, Tuple
from bert_score import score as bert_score
import torch
from pathlib import Path
DATA_ALIGN = Path("./models/alignscore")
DATA_ALIGN.mkdir(parents=True, exist_ok=True)
device = "mps" if torch.backends.mps.is_available() else ("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

/Users/jsoa/miniforge3/envs/alignscore-112/lib/python3.10/site-packages/textstat/textstat.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/jsoa/miniforge3/envs/alignscore-112/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mps


In [12]:
### Modelo requerido base, puede utilizarse large tambien, podria descargarse de HuggingFace, en una proxima revision lo ajusto.

!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt)
#!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-large.ckpt)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1333  100  1333    0     0   8220      0 --:--:-- --:--:-- --:--:--  8177
100 1875M  100 1875M    0     0  60.9M      0  0:00:30  0:00:30 --:--:-- 59.6M00:28 64.0M  0  60.5M      0  0:00:30  0:00:19  0:00:11 59.7M


In [4]:
def calcular_factualidad_alignscore(preds, refs,evaluation_mode, batch_size, device,flag_threshold: float = 0.5):
#evaluation_mode,    # 'nli_sp' (por defecto AlignScore), 'nli', 'bin_sp', 'bin'
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"

    # Import tardío para que esta función siga importando aunque no esté instalada la lib.
    from alignscore import AlignScore  
    # Inicializar scorer
    backbone = 'roberta-base'
    scorer = AlignScore(
        model="roberta-base",
        batch_size=batch_size,
        device=device,
        ckpt_path='models/alignscore/AlignScore-base.ckpt',
        evaluation_mode=evaluation_mode
    )

    scores = scorer.score(contexts=refs, claims=preds) 
    scores = [float(s) for s in scores]

    flags_low = [bool(s < flag_threshold) for s in scores]
    per_example = pd.DataFrame({"alignscore": scores,"flag_low": flags_low}) 

    summary = {
        "mean_alignscore": float(np.mean(scores)) if scores else float("nan"),
        "std_alignscore":  float(np.std(scores)) if scores else float("nan"),
        "min_alignscore":  float(np.min(scores)) if scores else float("nan"),
        "max_alignscore":  float(np.max(scores)) if scores else float("nan"),
        "n_examples":      int(len(scores)),
        "backbone":        backbone,
        "evaluation_mode": evaluation_mode,
        "batch_size":      int(batch_size),
        "device":          device,
        "ckpt_path":       'models/alignscore/AlignScore-base.ckpt',
        "flag_threshold":  float(flag_threshold)
    }

    return summary, per_example



In [5]:
def calcular_bertscore_relevancia(preds,refs,idf,rescale_with_baseline,batch_size,device):

    modelo = "roberta-base"
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"


    P, R, F1 = bert_score(
        cands=preds.tolist(),
        refs=refs.tolist(),
        lang='en',
        model_type=modelo,
        idf=idf,
        rescale_with_baseline=rescale_with_baseline,
        batch_size=batch_size,
        device=device
    )

    p_list = [float(p) for p in P]
    r_list = [float(r) for r in R]
    f1_list = [float(f) for f in F1]

    summary = {
        "mean_precision": float(np.mean(p_list)) if p_list else float("nan"),
        "mean_recall":    float(np.mean(r_list)) if r_list else float("nan"),
        "mean_f1":        float(np.mean(f1_list)) if f1_list else float("nan"),
        "backbone_for_bertscore": modelo,
        "idf": bool(idf),
        "rescale_with_baseline": bool(rescale_with_baseline),
        "batch_size": int(batch_size),
        "device": device if device is not None else "auto"
    }

    per_example = {
        "bertscore_precision": p_list,
        "bertscore_recall": r_list,
        "bertscore_f1": f1_list
    }

    per_example = pd.DataFrame(per_example)

    return summary, per_example


In [6]:
def calcular_legibilidad_textstat(preds):
    lang = 'en'
    textstat.set_lang(lang)
    rows = []
    for t in preds:
        t = t or ""

        row = {
            "flesch_reading_ease":  float(textstat.flesch_reading_ease(t)),
            "flesch_kincaid_grade": float(textstat.flesch_kincaid_grade(t)),
        }
        row.update({
            "gunning_fog":              float(textstat.gunning_fog(t)),
            "smog_index":               float(textstat.smog_index(t)) if textstat.sentence_count(t) >= 3 else float("nan"),
            "dale_chall":               float(textstat.dale_chall_readability_score(t)),
            "automated_readability":    float(textstat.automated_readability_index(t)),
            "coleman_liau":             float(textstat.coleman_liau_index(t)),
            "text_standard":            textstat.text_standard(t, float_output=True),
            "num_sentences":            int(textstat.sentence_count(t)),
            "num_words":                int(textstat.lexicon_count(t, removepunct=True)),
            "syllables":                int(textstat.syllable_count(t)),
            "reading_time_sec":         float(textstat.reading_time(t)),
        })

        rows.append(row)

    def _try_mean(key: str):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else float("nan")

    keys = sorted({k for r in rows for k in r.keys()})
    summary = {"n_examples": len(preds), "lang": lang}
    for k in keys:
        summary[f"mean_{k}"] = _try_mean(k)

    per_example = pd.DataFrame(rows) 
    return summary, per_example


In [ ]:
def calcular_metricas(df):
    print('**** Inicio relevancia')
    summary_relevancia, per_example_relevancia = calcular_bertscore_relevancia(
        df['gen_summary'],df['article'],
        idf=True,#Set pequeno a false, sino dejar en TRUE
        rescale_with_baseline=False,
        batch_size=1,
        device=device
    )
    print('**** Fin relevancia')
    print('**** Inicio legibilidad')
    summary_legibilidad, per_example_legibilidad = calcular_legibilidad_textstat(df['gen_summary'])
    print('**** Fin legibilidad')
    print('**** Inicio factualidad')
    summary_factualidad, per_example_factualidad = calcular_factualidad_alignscore(df['gen_summary'], df['article'],'nli_sp', 16, device)
    print('**** Fin factualidad')
    return summary_relevancia, summary_legibilidad,summary_factualidad


### Calculo de ejemplo de las 3 metricas requeridas para phi3.5

In [ ]:
device="cpu"
data = pd.read_csv('models/results/summaries_phi35.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_phi35.csv", index=False)

**** Inicio relevancia


/Users/jsoa/miniforge3/envs/alignscore-112/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of the model checkpoint at roberta-base were not used when initializing Ro

**** Fin factualidad
{'mean_precision': 0.8530013038923866, 'mean_recall': 0.837771927212414, 'mean_f1': 0.8450050744571184, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': 'cpu'}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 14.253421052631577, 'mean_coleman_liau': 15.244000000000003, 'mean_dale_chall': 9.989157894736842, 'mean_flesch_kincaid_grade': 10.433157894736842, 'mean_flesch_reading_ease': 49.641447368421055, 'mean_gunning_fog': 11.08286842105263, 'mean_num_sentences': 22.82894736842105, 'mean_num_words': 364.8, 'mean_reading_time_sec': 30.8485, 'mean_smog_index': 12.332786885245902, 'mean_syllables': 599.1578947368421, 'mean_text_standard': 11.260526315789473}
{'mean_alignscore': 0.45983980549009223, 'std_alignscore': 0.17386677445404883, 'min_alignscore': 0.1141374409198761, 'max_alignscore': 0.9775390028953552, 'n_examples': 380, 'backbone': 'roberta-base', 'evaluation_mode': 'nli_sp', 'batc